<a href="https://colab.research.google.com/github/kimheeseo/LSCNS/blob/main/COSMOS_Full_Transformer_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# OFC 2026: COSMOS 전체 데이터로 Transformer 학습 및 평가

이 노트북은 `PI_EDFA_Transformer/transformer_edfa.py`를 사용해 COSMOS feature/label CSV 전체로 모델을 학습하고 train/validation/test 분할 점수를 계산합니다. COSMOS에 Aging/SHB/Unseen 라벨이 없는 경우 해당 카테고리별 점수는 만들 수 없으며, 전체 COSMOS 점수를 보고합니다.

In [ ]:
import subprocess, sys, torch
from pathlib import Path
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.is_available())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'pandas', 'scikit-learn', 'matplotlib'], check=True)
def sh(args):
    print('+', ' '.join(map(str,args)))
    subprocess.run(list(map(str,args)), check=True)

PyTorch: 2.11.0+cu130 | GPU: True


## 1. 코드와 데이터 준비

Git LFS가 없는 Colab 환경에서는 자동 설치 후 버전을 확인합니다. 전체 LFS 대신 COSMOS feature/label CSV 두 파일만 다운로드합니다. CSV를 이미 가지고 있다면 아래 `FEATURES_CSV` 및 `LABELS_CSV` 경로를 지정할 수 있습니다.

In [ ]:
# Google Colab: install Git LFS before downloading the COSMOS CSV files
import shutil
if shutil.which('git-lfs') is None:
    sh(['bash', '-lc', 'apt-get update -qq && apt-get install -y git-lfs'])
sh(['git', 'lfs', 'version'])

+ bash -lc apt-get update -qq && apt-get install -y git-lfs
+ git lfs version


In [ ]:
repo = Path('/content/LSCNS')
data = Path('/content/ofc-ml-challenge-data-code')
if not (repo/'.git').exists():
    sh(['git', 'clone', '--depth', '1', 'https://github.com/kimheeseo/LSCNS.git', repo])
if not (data/'.git').exists():
    sh(['bash', '-lc', f'GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 https://github.com/ofc-ml-challenge/ofc-ml-challenge-data-code.git {data}'])

# Fetch only the two full COSMOS CSV files from Git LFS.
sh(['git', '-C', data, 'lfs', 'install', '--local'])
sh([
    'git', '-C', data, 'lfs', 'pull',
    '--include=Features/Train/COSMOS_features.csv,Features/Train/COSMOS_labels.csv'
])

FEATURES_CSV = None  # Optional: '/content/COSMOS_features.csv'
LABELS_CSV = None    # Optional: '/content/COSMOS_labels.csv'
fx = Path(FEATURES_CSV) if FEATURES_CSV else data / 'Features/Train/COSMOS_features.csv'
fy = Path(LABELS_CSV) if LABELS_CSV else data / 'Features/Train/COSMOS_labels.csv'
for f in (fx, fy):
    if not f.is_file():
        raise FileNotFoundError(f'COSMOS CSV not found: {f}')
    with f.open('rb') as stream:
        head = stream.read(128)
    if head.startswith(b'version https://git-lfs'):
        raise RuntimeError(f'Git LFS pointer only; CSV download failed: {f}')
print('Features:', fx, '\nLabels:', fy)

+ git -C /content/ofc-ml-challenge-data-code lfs install --local
+ git -C /content/ofc-ml-challenge-data-code lfs pull --include=Features/Train/COSMOS_features.csv,Features/Train/COSMOS_labels.csv
Features: /content/ofc-ml-challenge-data-code/Features/Train/COSMOS_features.csv 
Labels: /content/ofc-ml-challenge-data-code/Features/Train/COSMOS_labels.csv


## 2. 실행 설정

학습 실행은 이 노트북을 실행할 때 시작됩니다. 기본적으로 GPU가 있으면 사용합니다.

In [ ]:
MODEL_DIR=repo/'26OFC_ML_challenge/PI_EDFA_Transformer'
OUTPUT=Path('/content/cosmos_transformer_results')
EPOCHS=250
TEST_FRACTION=0.15
VALIDATION_FRACTION=0.15
SEED=2026
print('Model directory:',MODEL_DIR,'| output:',OUTPUT)

Model directory: /content/LSCNS/26OFC_ML_challenge/PI_EDFA_Transformer | output: /content/cosmos_transformer_results


In [ ]:
# The script uses the repository's model and official_score implementation.
runner=MODEL_DIR/'test_transformer_edfa.py'
assert runner.exists(), f'Missing evaluation script: {runner}'
sh([sys.executable,runner,'--data-root',data,'--features',fx,'--labels',fy,'--output',OUTPUT,
    '--epochs',str(EPOCHS),'--test-frac',str(TEST_FRACTION),
    '--val-frac',str(VALIDATION_FRACTION),'--seed',str(SEED)])

+ /usr/bin/python3 /content/LSCNS/26OFC_ML_challenge/PI_EDFA_Transformer/test_transformer_edfa.py --data-root /content/ofc-ml-challenge-data-code --features /content/ofc-ml-challenge-data-code/Features/Train/COSMOS_features.csv --labels /content/ofc-ml-challenge-data-code/Features/Train/COSMOS_labels.csv --output /content/cosmos_transformer_results --epochs 250 --test-frac 0.15 --val-frac 0.15 --seed 2026


## 3. 결과 확인

JSON에는 전체 공식 점수와 MAE, 표준편차, P95, 최대오차 및 보조 지표가 저장됩니다. 범주 라벨이 원본 데이터에 있을 때만 카테고리별 결과가 표시됩니다.

In [ ]:
import json, pandas as pd
from IPython.display import display
report=json.loads((OUTPUT/'cosmos_split_scores.json').read_text())
display(pd.DataFrame([{'split':s,**report[s]['overall']} for s in ('validation','test')]).round(6))
print(json.dumps(report,indent=2,ensure_ascii=False)[:12000])
print('Saved artifacts:', *[p.name for p in OUTPUT.iterdir()])

,split,score,mae,std,p95,max_error,penalty_p95,penalty_max,samples,loaded_points,rmse,mean_bias,median_absolute_error,p99_absolute_error,within_0p1_db,within_0p2_db,within_0p5_db
0,validation,0.083513,0.070819,0.034503,0.173276,0.948463,0.0,0.075187,17675,431366,0.093244,0.000220,0.049524,0.359177,0.807099,0.964548,0.997860
1,test,0.093633,0.070466,0.034339,0.168787,1.048941,0.0,0.180154,20794,507180,0.090223,0.000993,0.049234,0.329211,0.810308,0.968140,0.998279


{
  "rows": {
    "total": 138624,
    "train": 100155,
    "validation": 17675,
    "test": 20794
  },
  "split": {
    "seed": 2026,
    "test_fraction": 0.15,
    "validation_fraction_of_trainval": 0.15
  },
  "validation": {
    "overall": {
      "score": 0.08351338321938029,
      "mae": 0.07081920006212238,
      "std": 0.03450289280131566,
      "p95": 0.17327594757080078,
      "max_error": 0.9484634399414062,
      "penalty_p95": 0.0,
      "penalty_max": 0.07518749237060551,
      "samples": 17675,
      "loaded_points": 431366,
      "rmse": 0.09324444361056801,
      "mean_bias": 0.000220272000643372,
      "median_absolute_error": 0.04952430725097656,
      "p99_absolute_error": 0.3591767311096181,
      "within_0p1_db": 0.8070988441369974,
      "within_0p2_db": 0.964547507221246,
      "within_0p5_db": 0.9978602856970646
    },
    "by_category": {
      "COSMOS": {
        "score": 0.08351338321938029,
        "mae": 0.07081920006212238,
        "std": 0.03450289280131